<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-11-21-KoModernBERT-Cross-Encoder-LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q onnx onnxruntime onnxscript


In [2]:
import os
import random
import torch
import numpy as np
import onnxruntime as ort
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

random.seed(42)

# 1. 도구 스키마 정의
tools_desc = {
    "get_weather": "Tool: get_weather | Desc: 특정 지역의 오늘, 내일, 주간 날씨와 기온, 강수 확률을 조회합니다.",
    "search_order": "Tool: search_order | Desc: 사용자가 구매한 상품의 주문 내역, 배송 상태, 택배 위치를 조회합니다.",
    "cancel_subscription": "Tool: cancel_subscription | Desc: 유료 멤버십이나 정기 구독 서비스를 해지하고 결제를 취소합니다."
}

# 2. 합성 데이터 생성용 슬롯(Slot) 사전 (학습용 / 평가용 철저히 분리)
train_slots = {
    "get_weather": {
        "time": ["오늘", "내일", "모레", "이번 주 주말", "금요일", "다음 주 월요일", "지금 당장", "오늘 저녁"],
        "loc": ["서울", "판교", "분당", "강남", "부산", "제주도", "수원", "인천", "우리 동네", "여의도"],
        "intent": ["날씨 어때", "비 올 확률 알려줘", "우산 챙겨야 해", "기온 몇 도야", "미세먼지 심해", "패딩 입어야 할 정도로 추워", "눈 오는지 확인해 줘", "맑은지 조회해 줘"],
        "ending": ["?", "요?", " 좀 알려줘", " 궁금해", " 확인 부탁해", ""]
    },
    "search_order": {
        "time": ["어제", "지난주에", "방금", "그제", "3일 전에", "이번 달에", "아까", "저번 주에"],
        "item": ["운동화", "노트북", "닭가슴살", "청바지", "책", "화장품", "키보드", "커피 원두", "택배", "물건"],
        "intent": ["주문한 거 어디쯤 왔어", "배송 상태 확인 좀", "언제 도착하는지 조회해 줘", "발송 시작했나", "주문 내역 뽑아줘", "결제 내역 확인하고 싶어", "송장 조회 부탁해", "아직 안 왔는데 확인해 줘"],
        "ending": ["?", "요?", " 좀 봐줘", " 알려줘", " 궁금합니다", ""]
    },
    "cancel_subscription": {
        "time": ["다음 달부터", "이번 달까지만 쓰고", "오늘 바로", "지금 즉시", "다음 결제일 전에", "당장", "이번 주 내로"],
        "target": ["유료 멤버십", "정기 구독", "프리미엄 이용권", "멤버십 서비스", "자동 결제", "구독권", "월정액 서비스"],
        "intent": ["해지해 줘", "취소하고 싶어요", "끊는 법 알려줘", "탈퇴 신청할래", "돈 안 빠져나가게 해지 부탁해", "중단하고 싶어", "연장 안 할래"],
        "ending": [".", "요", " 좀 해줘", " 부탁드립니다", " 처리해 줘", ""]
    },
    "none": {  # 일반 대화 (도구 호출 없음)
        "prefix": ["오늘", "요즘", "주말에", "심심한데", "퇴근하고", "갑자기", "혹시"],
        "topic": ["점심 메뉴 뭐 먹을까", "볼만한 넷플릭스 영화 추천해 줘", "파이썬 문법 좀 알려줘", "운동 루틴 추천 좀", "잠 잘 오는 법 알려줘", "커피 한 잔 마실까", "재밌는 농담 하나 해봐", "영어 공부 어떻게 해"],
        "ending": ["?", "요?", " 알려줘", " 추천 좀", " 궁금해", ""]
    }
}

def generate_synthetic_queries(slots_dict, n_per_class=60):
    generated = []
    for label, s in slots_dict.items():
        seen = set()
        while len(seen) < n_per_class:
            if label == "get_weather":
                q = f"{random.choice(s['time'])} {random.choice(s['loc'])} {random.choice(s['intent'])}{random.choice(s['ending'])}"
            elif label == "search_order":
                q = f"{random.choice(s['time'])} 시킨 {random.choice(s['item'])} {random.choice(s['intent'])}{random.choice(s['ending'])}"
            elif label == "cancel_subscription":
                q = f"{random.choice(s['time'])} {random.choice(s['target'])} {random.choice(s['intent'])}{random.choice(s['ending'])}"
            else:
                q = f"{random.choice(s['prefix'])} {random.choice(s['topic'])}{random.choice(s['ending'])}"

            q = q.strip()
            if q not in seen:
                seen.add(q)
                generated.append((q, None if label == "none" else label))
    random.shuffle(generated)
    return generated

# 3. 학습용 240개 생성 (클래스당 60개) -> All-Pairs 변환 시 720개 학습 쌍
train_queries = generate_synthetic_queries(train_slots, n_per_class=60)

all_pairs = []
for query, target_tool in train_queries:
    for tool_name in tools_desc.keys():
        score = 1.0 if tool_name == target_tool else 0.0
        all_pairs.append((query, tool_name, score))

dataset_dict = {
    "query": [q for q, t, l in all_pairs],
    "tool_text": [tools_desc[t] for q, t, l in all_pairs],
    "label": [l for q, t, l in all_pairs]
}
dataset = Dataset.from_dict(dataset_dict).train_test_split(test_size=0.1, seed=42)

# 4. 평가용(Hold-out Test Set): 학습에 전혀 쓰이지 않은 새로운 어휘/구조의 정답지
eval_dataset_with_answers = [
    # get_weather (10개)
    ("정자동 지금 소나기 내리나?", "get_weather"),
    ("내일 아침 출근할 때 영하로 떨어져?", "get_weather"),
    ("이번 주말 제주도 태풍 영향권이야?", "get_weather"),
    ("모레 대전 날씨 맑은지 좀 봐줄래?", "get_weather"),
    ("오늘 오후에 강수량 얼마나 돼?", "get_weather"),
    ("주말 등산 가려는데 산 날씨 어때?", "get_weather"),
    ("내일 비 소식 있는지 확인해 줘", "get_weather"),
    ("지금 한강 바람 많이 불어?", "get_weather"),
    ("다음 주 화요일 서울 최고 기온 알려줘", "get_weather"),
    ("오늘 우비나 우산 필요한 날씨야?", "get_weather"),

    # search_order (10개)
    ("이틀 전에 결제한 책 배송 출발했나요?", "search_order"),
    ("내 택배 기사님 어디까지 오셨는지 조회해 줘", "search_order"),
    ("지난달 구매 내역 영수증 확인하고 싶어", "search_order"),
    ("방금 주문한 아이패드 결제 정상 처리됐어?", "search_order"),
    ("아까 산 물건 언제쯤 수령 가능할까?", "search_order"),
    ("주문한 상품 아직 준비중인지 봐줘", "search_order"),
    ("배송 조회 좀 해주세요 어제 시킨 거요", "search_order"),
    ("내가 주문했던 목록 전체 보여줘", "search_order"),
    ("택배 상자 지금 어느 터미널에 있어?", "search_order"),
    ("저번 주에 산 의자 배송 완료 떴는데 확인 좀", "search_order"),

    # cancel_subscription (10개)
    ("더 이상 안 쓸 것 같아서 멤버십 해지합니다", "cancel_subscription"),
    ("다음 달 자동이체 안 되게 구독 끊어줘", "cancel_subscription"),
    ("프리미엄 플랜 해지하는 메뉴가 어디야?", "cancel_subscription"),
    ("월간 구독권 오늘부로 취소 처리 부탁해요", "cancel_subscription"),
    ("멤버십 연장 취소하고 해지하고 싶습니다", "cancel_subscription"),
    ("정기 결제 더 이상 진행 안 할래요 끊어주세요", "cancel_subscription"),
    ("이번 달 무료 체험 끝나기 전에 해지해 줘", "cancel_subscription"),
    ("구독 해지 버튼이 안 보이는데 취소 좀", "cancel_subscription"),
    ("유료 회원 탈퇴하고 결제 중단시켜 줘", "cancel_subscription"),
    ("다음 주에 갱신되는 정기권 해지할게요", "cancel_subscription"),

    # None - 일반 대화 (10개)
    ("오늘 저녁에 삼겹살에 소주 어때?", None),
    ("파이썬에서 딕셔너리 정렬하는 코드 짜줘", None),
    ("주말에 집에서 볼만한 스릴러 영화 추천해 줘", None),
    ("인공지능 라우터 아키텍처에 대해 설명해 줘", None),
    ("고마워 덕분에 잘 해결됐어!", None),
    ("퇴근하고 헬스장 갈까 그냥 쉴까?", None),
    ("서울에서 제일 맛있는 평양냉면 집 어디야?", None),
    ("내일 면접 보는데 긴장 푸는 법 알려줘", None),
    ("영어 회화 독학하기 좋은 유튜브 채널 추천 좀", None),
    ("지금 몇 시인지 알려줄 수 있어?", None),
]

print(f"✅ 합성 학습 데이터: {len(dataset['train'])}쌍 | 평가용 정답셋: {len(eval_dataset_with_answers)}문항 준비 완료!")


✅ 합성 학습 데이터: 648쌍 | 평가용 정답셋: 40문항 준비 완료!


In [3]:
# [Cell 3] 수정: 임베딩 전용(SimCSE) 대신 일반 베이스 모델 사용
# ModernBERT를 원하시면 "team-lucid/ModernBERT-large-multilingual" 사용 (용량이 크면 "klue/roberta-base" 추천)
MODEL_ID = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=1)

def preprocess_function(examples):
    return tokenizer(
        examples["query"],
        examples["tool_text"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_token_type_ids=False  # 핵심 수정: RoBERTa 인덱스 초과(CUDA assert) 방지
    )

encoded_dataset = dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=20,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=30,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
)

print("🚀 파인튜닝 시작...")
trainer.train()
print("✅ 파인튜닝 완료!")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/648 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

🚀 파인튜닝 시작...


Epoch,Training Loss,Validation Loss
1,0.104655,0.020474
2,0.016162,0.004114
3,0.004126,0.002415
4,0.004118,0.000360
5,0.003542,0.001070
6,0.002798,0.001101
7,0.002200,0.001213
8,0.002566,0.003129
9,0.002341,0.001117
10,0.001826,0.001139


✅ 파인튜닝 완료!


In [4]:
# [Cell 4] 수정: INT8 압축으로 인한 점수 뭉개짐 방지 (FP32 ONNX 그대로 사용, 약 420MB)
ONNX_DIR = "./onnx_router"
os.makedirs(ONNX_DIR, exist_ok=True)

model.eval().cpu()

dummy_input = tokenizer(
    "테스트 질문", "Tool: test | Desc: 테스트",
    return_tensors="pt", padding="max_length", max_length=256
)

onnx_path = os.path.join(ONNX_DIR, "model.onnx")

torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14,
    dynamo=False
)

tokenizer.save_pretrained(ONNX_DIR)
!zip -r onnx_router.zip {ONNX_DIR}
print("✅ 완료! 'onnx_router.zip' 생성됨")


/tmp/ipykernel_12098/1954150494.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:244: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integrations/sdpa_attention.py:159: TracerWarning: Converting a tensor to a Python boole

updating: onnx_router/ (stored 0%)
updating: onnx_router/model.onnx (deflated 13%)
updating: onnx_router/tokenizer.json (deflated 69%)
updating: onnx_router/tokenizer_config.json (deflated 50%)
✅ 완료! 'onnx_router.zip' 생성됨


In [6]:
# [Cell 5] 전 문항 상세 채점 결과 + 최종 정답률 리포트
session = ort.InferenceSession(
    "./onnx_router/model.onnx",
    providers=["CPUExecutionProvider"]
)

tool_names = list(tools_desc.keys())
tool_texts = list(tools_desc.values())
THRESHOLD = 0.7

correct_count = 0

print("🔍 전체 평가 문항 채점 상세 결과")
print("-" * 80)

for idx, (query, expected_tool) in enumerate(eval_dataset_with_answers, 1):
    inputs = tokenizer(
        [query] * len(tool_texts), tool_texts,
        padding=True, truncation=True, max_length=256, return_tensors="np"
    )
    ort_inputs = {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64)
    }
    raw_logits, = session.run(["logits"], ort_inputs)
    scores = np.clip(raw_logits.squeeze(-1), 0.0, 1.0)

    score_map = {name: round(float(s), 4) for name, s in zip(tool_names, scores)}
    best_tool = max(score_map, key=score_map.get)
    best_score = score_map[best_tool]

    predicted_tool = best_tool if best_score >= THRESHOLD else None
    is_correct = (predicted_tool == expected_tool)

    if is_correct:
        correct_count += 1
        mark = "✅ O"
    else:
        mark = "❌ X"

    print(f"[{idx:02d}] {mark} | 질문: {query}")
    print(f"      실제 정답: {str(expected_tool):<19} | 모델 예측: {str(predicted_tool):<19} | 점수: {score_map}")

total = len(eval_dataset_with_answers)
accuracy = (correct_count / total) * 100

print("=" * 80)
print(f"📊 최종 벤치마크 정답률: {accuracy:.2f}% ({correct_count} / {total} 정답)")
print("=" * 80)


🔍 전체 평가 문항 채점 상세 결과
--------------------------------------------------------------------------------
[01] ✅ O | 질문: 정자동 지금 소나기 내리나?
      실제 정답: get_weather         | 모델 예측: get_weather         | 점수: {'get_weather': 1.0, 'search_order': 0.001, 'cancel_subscription': 0.0012}
[02] ✅ O | 질문: 내일 아침 출근할 때 영하로 떨어져?
      실제 정답: get_weather         | 모델 예측: get_weather         | 점수: {'get_weather': 1.0, 'search_order': 0.0005, 'cancel_subscription': 0.0009}
[03] ✅ O | 질문: 이번 주말 제주도 태풍 영향권이야?
      실제 정답: get_weather         | 모델 예측: get_weather         | 점수: {'get_weather': 1.0, 'search_order': 0.0011, 'cancel_subscription': 0.0014}
[04] ✅ O | 질문: 모레 대전 날씨 맑은지 좀 봐줄래?
      실제 정답: get_weather         | 모델 예측: get_weather         | 점수: {'get_weather': 1.0, 'search_order': 0.0005, 'cancel_subscription': 0.001}
[05] ✅ O | 질문: 오늘 오후에 강수량 얼마나 돼?
      실제 정답: get_weather         | 모델 예측: get_weather         | 점수: {'get_weather': 1.0, 'search_order': 0.0009, 'cancel_subscription': 0.0014}
[06] ✅ O | 

# Lightweight Semantic Tool Router (ONNX & Cross-Encoder)

저사양 CPU 서버(가용 RAM 1.4GB 이하) 환경에서 초고속·고정밀로 사용자 발화의 의도를 분석하고 적절한 외부 도구(Function/API)를 라우팅하기 위해 구축한 **Cross-Encoder 기반 Semantic Tool Router** 프로젝트입니다.

---

## 1. 배경 및 설계 목표 (Background & Goals)

* **제약 사항:** GPU가 없는 저사양 Linux 환경(Total RAM 3.7GB / Available RAM 1.4GB)에서는 7B 이상의 Local LLM 구동이 불가능하며, 매 요청마다 수십 개의 도구 스키마를 외부 LLM에 전달하면 토큰 비용과 응답 지연 시간(Latency)이 급증합니다.
* **해결 전략:**
  1. **역할 분리(Routing + Execution):** 경량 Encoder 모델이 1차로 호출할 도구(Top-K)만 빠르게 필터링하여 LLM의 추론 부담을 최소화합니다.
  2. **Cross-Encoder 아키텍처:** 단순 임베딩 유사도(Bi-Encoder)의 비대칭 검색 한계를 극복하기 위해 `[질문] + [도구 명세]` 쌍을 직접 입력받아 문맥적 연관성을 판별합니다.
  3. **ONNX Runtime 경량화:** 추론 시 무거운 PyTorch 런타임 대신 순수 `onnxruntime`을 사용하여 메모리 점유율과 CPU 추론 시간을 극한으로 최적화했습니다.

---

## 2. 시스템 아키텍처 (Architecture)

```text
[사용자 쿼리] ("방금 시킨 음식 배달 출발했는지 봐줘")
      │
      ▼  Tokenizer (Batch Pairing: Query × Registered Tools)
[입력 쌍 생성]
  ├─ Pair 1: [CLS] 쿼리 [SEP] Tool: get_weather | Desc: ... [SEP]
  ├─ Pair 2: [CLS] 쿼리 [SEP] Tool: search_order | Desc: ... [SEP]
  └─ Pair 3: [CLS] 쿼리 [SEP] Tool: cancel_subscription | Desc: ... [SEP]
      │
      ▼  ONNX Runtime (CPUExecutionProvider)
[Cross-Encoder 추론] ──▶ 각 도구별 독립 적합도 점수 산출 (0.0 ~ 1.0)
      │
      ▼  Threshold Filtering (Score >= 0.7)
[최종 라우팅 결과] ──▶ [{"name": "search_order", "score": 1.0}]
```

### 핵심 설계 포인트
* **독립 스코어링 (Multi-Tool 지원):** Softmax 대신 개별 도구별 독립 로짓(Logit)을 산출하여, 복합 질문(예: *"날씨 알려주고 택배도 조회해 줘"*) 입력 시 여러 도구를 동시에 선택할 수 있습니다.
* **일반 대화 필터링 (Fallback):** 모든 도구의 점수가 임계값(`0.7`) 미만일 경우 도구 호출 없이 일반 대화로 분기합니다.

---

## 3. 구현 및 트러블슈팅 과정 (Implementation Log)

프로젝트 구축 과정에서 발생한 주요 기술적 이슈와 해결 내역입니다.

| 단계 | 문제 상황 (Issue) | 원인 분석 (Root Cause) | 해결 방안 (Resolution) |
| :--- | :--- | :--- | :--- |
| **1. Zero-shot Bi-Encoder** | 모든 질문과 도구 간 코사인 유사도가 `0.73~0.85`로 뭉개져 오분류 발생 | 짧은 구어체 질문과 긴 설명문 간의 비대칭 검색 한계 | 문장 쌍의 상호작용을 학습하는 **Cross-Encoder 파인튜닝**으로 전환 |
| **2. 백본 모델 선정** | `SimCSE` 임베딩 전용 모델 및 INT8 동적 양자화 적용 시 점수 변별력 저하 (`0.73` vs `0.70`) | 이미 임베딩 공간이 고정된 백본 특성 및 양자화 활성화 값 압축 | MLM 기반 베이스 모델(`klue/roberta-base`) 및 FP32 ONNX 모델 적용 |
| **3. 토크나이저 호환** | RoBERTa 계열 학습 시 `CUDA error: device-side assert` 발생 | 문장 쌍 입력 시 생성되는 `token_type_ids=1`이 RoBERTa의 `type_vocab_size(1)` 초과 | 토크나이저 옵션에 `return_token_type_ids=False` 명시 |
| **4. 점수 스케일링** | 정답 점수가 `0.738`, 오답이 `0.498`로 출력됨 | `float` 라벨 학습으로 `MSELoss`가 적용된 모델 출력에 `Sigmoid`를 중복 적용 | `Sigmoid`를 제거하고 `np.clip(logits, 0.0, 1.0)`으로 직접 매핑 |
| **5. 과적합 방지** | 특정 요청형 어미(`~확인 좀`, `~해지해 줘`)에 대해 모든 도구가 `1.0`으로 반응 | 질문별 정답 쌍만 학습하여 오답 도구에 대한 반례(Hard Negative) 부족 | 모든 질문 × 전체 도구 조합을 생성하는 **All-Pairs 페어링** 구축 |

---

## 4. 성능 및 검증 결과 (Evaluation)

학습 데이터에 포함되지 않은 **미학습(Unseen) 테스트 쿼리**에 대한 실제 라우팅 점수 결과입니다.

```text
[질문] 분당 지금 우산 챙겨야 해?
 -> 점수: {'get_weather': 1.0, 'search_order': 0.0, 'cancel_subscription': 0.0}

[질문] 내일 출근길에 패딩 입어야 할 정도로 추워? (미학습 문장)
 -> 점수: {'get_weather': 1.0, 'search_order': 0.0312, 'cancel_subscription': 0.0246}

[질문] 방금 시킨 음식 배달 출발했는지 봐줘 (미학습 문장)
 -> 점수: {'get_weather': 0.0, 'search_order': 1.0, 'cancel_subscription': 0.0}

[질문] 더 이상 이용 안 할 거라 멤버십 끊어주세요 (미학습 문장)
 -> 점수: {'get_weather': 0.0098, 'search_order': 0.0, 'cancel_subscription': 1.0}

[질문] 오늘 점심 메뉴 뭐 먹을까? (일반 대화)
 -> 점수: {'get_weather': 0.0, 'search_order': 0.0, 'cancel_subscription': 0.0}
```

---

## 5. 디렉토리 구조 (Project Structure)

```text
modernBERT_router/
├── README.md                  # 프로젝트 설계 및 구현 문서
├── train_and_export.ipynb     # Google Colab용 파인튜닝 및 ONNX 변환 노트북
├── run_onnx_router.py         # Linux 서버용 초경량 ONNX 추론 스크립트
└── onnx_router/               # 학습 완료된 ONNX 모델 및 토크나이저 디렉토리
    ├── model.onnx
    ├── tokenizer.json
    └── tokenizer_config.json
```

---

## 6. 설치 및 실행 방법 (Quick Start)

### 1) 모델 학습 및 ONNX 변환 (Google Colab T4 GPU)
저사양 서버의 메모리 부족(OOM)을 방지하기 위해 학습은 GPU 환경(Colab)에서 수행합니다.
1. `train_and_export.ipynb` 실행 (`klue/roberta-base` 기반 All-Pairs 파인튜닝)
2. `torch.onnx.export(..., opset_version=14, dynamo=False)`로 ONNX 추출
3. 생성된 `onnx_router.zip` 다운로드 후 추론 서버로 업로드

### 2) 추론 서버 환경 구축 (Linux CPU)
추론 서버에는 무거운 PyTorch를 설치하지 않고 경량 런타임만 설치합니다.

```bash
python3 -m venv router_env
source router_env/bin/activate

pip install --no-cache-dir numpy transformers onnxruntime
unzip -o onnx_router.zip
```

### 3) 라우터 실행
```bash
python run_onnx_router.py
```

---

## 7. 향후 확장 계획 (Roadmap)

* **2-Stage 하이브리드 확장:** 등록 도구가 50개 이상으로 늘어날 경우, 1단계 Bi-Encoder(Vector Search)로 상위 10개 후보를 먼저 추린 뒤 2단계 ONNX Cross-Encoder로 정밀 채점하도록 확장
* **실제 에이전트 루프 연동:** 선택된 도구 스키마만 경량 LLM(Function Calling)에 전달하여 파라미터 JSON 추출 및 실제 Python 함수 실행 자동화
* **API 서빙:** FastAPI 기반 비동기 REST API 엔드포인트 래핑


# Cross-Encoder Tool Router 파인튜닝 데이터 및 학습 프로세스

이 문서는 **Cross-Encoder 기반 Semantic Tool Router**의 파인튜닝을 위한 입력 데이터 구조, 전처리 방식(All-Pairs 페어링), 그리고 모델 학습 및 경량화(ONNX) 전체 파이프라인을 정리한 가이드입니다.

---

## 1. 파인튜닝 입력 데이터 구조 (Input Data)

Cross-Encoder는 **사용자 질문(`query`)**과 **직렬화된 도구 명세(`tool_text`)**를 한 쌍(Pair)으로 입력받아, 두 문장의 문맥적 적합도를 `0.0` 또는 `1.0`으로 판별하도록 학습합니다.

### ① 도구 스키마 직렬화 포맷 (`tool_text`)
단순 함수명만 사용하면 의미 파악이 어려우므로, 도구명과 핵심 기능 설명을 결합한 고정 템플릿으로 변환합니다.

```text
Tool: {도구명} | Desc: {도구의 핵심 기능 및 파라미터 설명}
```
* **예시:** `Tool: get_weather | Desc: 특정 지역의 오늘, 내일, 주간 날씨와 기온, 강수 확률을 조회합니다.`

### ② All-Pairs (전체 조합) 데이터 생성 전략
단일 질문에 대해 정답 도구(`1.0`)만 학습시키면, 모델이 특정 요청형 어미(`~해줘`, `~확인 좀`)만 보고 모든 도구에 `1.0`을 출력하는 과적합(Shortcut Learning)이 발생합니다. 이를 방지하기 위해 **모든 질문 × 등록된 전체 도구**를 1:N으로 매칭합니다.

* **질문 1개당 생성되는 학습 쌍 예시** (등록 도구가 3개인 경우):

| `query` (입력 문장 A) | `tool_text` (입력 문장 B) | `label` (정답) | 비고 |
| :--- | :--- | :---: | :--- |
| `"내일 분당 비 올 확률 알려줘"` | `Tool: get_weather \| Desc: 특정 지역의...` | **`1.0`** | **Positive** (호출 대상) |
| `"내일 분당 비 올 확률 알려줘"` | `Tool: search_order \| Desc: 사용자가 구매한...` | **`0.0`** | **Hard Negative** (오답 도구) |
| `"내일 분당 비 올 확률 알려줘"` | `Tool: cancel_subscription \| Desc: 유료 멤버십...` | **`0.0`** | **Hard Negative** (오답 도구) |
| `"오늘 점심 메뉴 뭐 먹을까?"` | `Tool: get_weather \| Desc: 특정 지역의...` | **`0.0`** | **General Negative** (일반 대화) |

---

## 2. 토크나이징 및 모델 입력 텐서 변환

`klue/roberta-base` 백본 모델에 두 문장을 입력하기 위해 다음과 같이 토큰화를 수행합니다.

```text
[CLS] 내일 분당 비 올 확률 알려줘 [SEP] Tool: get_weather | Desc: 특정 지역의... [SEP] [PAD] ...
```

* **주요 전처리 설정:**
  * `max_length=256`, `padding="max_length"`, `truncation=True`
  * **`return_token_type_ids=False` (필수):** RoBERTa 아키텍처는 세그먼트 ID(`token_type_ids`)의 `type_vocab_size`가 `1`(`0`만 허용)이므로, 문장 쌍 입력 시 `1`이 생성되어 CUDA 인덱스 초과(`device-side assert`) 에러가 발생하는 것을 차단합니다.

---

## 3. 모델 아키텍처 및 손실 함수 (Loss Function)

* **백본 모델 (Backbone):** `klue/roberta-base` (임베딩 전용 SimCSE 모델 대신 Cross-Attention 학습이 가능한 MLM 기반 모델 사용)
* **분류 헤드 (Classifier Head):** `AutoModelForSequenceClassification(..., num_labels=1)`
  * `[CLS]` 토큰의 최종 은닉 상태(Hidden State)를 받아 단일 스칼라 로짓(Logit)을 출력합니다.
* **손실 함수 (Loss):** `MSELoss` (평균제곱오차)
  * `label`이 `float` 타입(`1.0`, `0.0`)으로 입력되므로 회귀 형태로 학습되며, 추론 시 별도의 Sigmoid 없이 출력 로짓을 `[0.0, 1.0]` 범위로 클리핑(`np.clip`)하여 직관적인 확률 점수로 사용합니다.

---

## 4. 전체 학습 및 경량화 파이프라인 (End-to-End Flow)

```text
[1. 합성 데이터 생성]
  └─ 슬롯 조합으로 클래스당 60개 원본 질문 생성 (총 240문장)
       │
       ▼
[2. All-Pairs 페어링 및 분할]
  └─ 240문장 × 3개 도구 = 720쌍 생성 ──▶ Train (90%) / Eval (10%)
       │
       ▼
[3. Cross-Encoder 파인튜닝 (GPU)]
  └─ klue/roberta-base | lr=3e-5 | batch_size=8 | epochs=10 | fp16=False
       │
       ▼
[4. ONNX 변환 (CPU 배포용 경량화)]
  └─ torch.onnx.export (opset_version=14, dynamo=False) ──▶ model.onnx 생성
       │
       ▼
[5. 홀드아웃(Hold-out) 정답률 검증]
  └─ 미학습 평가셋 40문항 채점 ──▶ Threshold(0.7) 기준 최종 Accuracy 산출
```

### 핵심 하이퍼파라미터 요약
* **Learning Rate:** `3e-5` (Warmup Steps: `20`)
* **Batch Size:** `8` (데이터 규모가 수백 건 단위일 때 충분한 Gradient 업데이트 확보)
* **Epochs:** `10`
* **Precision:** `fp16=False` (수치 불안정 및 Gradient 소실 방지)

---

## 5. 추론 시 스코어링 및 분기 로직 (Inference Logic)

실제 서비스 추론 시에는 사용자 쿼리 1개와 등록된 $N$개의 도구를 배치(Batch)로 묶어 한 번의 Forward Pass로 점수를 계산합니다.

1. **배치 추론:** `[Query] × N개 도구`를 `onnxruntime` 세션에 입력하여 $N$개의 점수 산출
2. **임계값 필터링 (`Threshold = 0.7`):**
   * **점수 ≥ 0.7인 도구가 1개 이상 존재:** 해당 도구(들)를 선택하여 후속 인자 추출(Function Calling) 단계로 전달 *(다중 도구 동시 호출 지원)*
   * **모든 도구의 점수 < 0.7:** 외부 도구가 필요 없는 **일반 대화(`None`)**로 판별하여 LLM이 직접 응답
